In [ ]:
# Instalação do pacote
# !pip install -q -U google-genai

from google import genai
from google.genai import types
import json

# Importa a biblioteca de segredos do Colab
from google.colab import userdata

# Puxa a chave da API do cofre do Colab (Será enviado orientações de configuração e chave na plataforma, por questões de segurança)
API_KEY = userdata.get('GEMINI_API_KEY')

# Inicialização do client
client = genai.Client(api_key=API_KEY)

#Check da autenticação da API
print("Client autenticado com sucesso e chave protegida!")

Client autenticado com sucesso e chave protegida!


In [ ]:
# Simulando a anotação de um médico em um prontuário (texto)

texto_clinico_simulado = """
Paciente deu entrada no PS às 14h30 relatando dor no peito irradiando para o braço esquerdo,
com início há cerca de 45 minutos. Nega episódios anteriores semelhantes.
Sinais vitais no momento da triagem: Pressão arterial 150/95 mmHg, Frequência cardíaca 110 bpm.
Paciente relata ser hipertenso e fazer uso de Losartana 50mg, mas esqueceu de tomar hoje.
Solicito ECG de urgência e exames de enzimas cardíacas.
"""

print("Texto Original:\n", texto_clinico_simulado)

Texto Original:
 
Paciente deu entrada no PS às 14h30 relatando dor no peito irradiando para o braço esquerdo, 
com início há cerca de 45 minutos. Nega episódios anteriores semelhantes. 
Sinais vitais no momento da triagem: Pressão arterial 150/95 mmHg, Frequência cardíaca 110 bpm. 
Paciente relata ser hipertenso e fazer uso de Losartana 50mg, mas esqueceu de tomar hoje. 
Solicito ECG de urgência e exames de enzimas cardíacas.



In [ ]:
# O prompt descrevendo o que a IA deve fazer

prompt = f"""
Você é um assistente de IA especializado em cardiologia e estruturação de dados clínicos.
Sua tarefa é ler o texto livre do prontuário médico abaixo e extrair as informações relevantes.

Retorne APENAS um objeto JSON válido, sem formatações markdown (sem ```json), com as seguintes chaves:
- "sintomas_principais": (lista de strings)
- "pressao_arterial": (string)
- "frequencia_cardiaca": (string)
- "historico_doencas": (lista de strings)
- "medicamentos_em_uso": (lista de strings)
- "procedimentos_solicitados": (lista de strings)
- "nivel_urgencia": (string: "Baixo", "Médio" ou "Alto" - deduzido pelo contexto clínico)

Texto do Prontuário:
{texto_clinico_simulado}
"""

# Chamada CORRETA para a nova biblioteca google.genai
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.1
    )
)

In [ ]:
# Carregando a resposta gerada como um dicionário Python real
try:
    dados_estruturados = json.loads(response.text)

    print("--- DADOS EXTRAÍDOS COM SUCESSO ---\n")
    # Imprimindo de forma formatada (Pretty Print)
    print(json.dumps(dados_estruturados, indent=4, ensure_ascii=False))

except json.JSONDecodeError as e:
    print("Erro ao decodificar o JSON:", e)
    print("Resposta bruta da IA:", response.text)

--- DADOS EXTRAÍDOS COM SUCESSO ---

{
    "sintomas_principais": [
        "dor no peito irradiando para o braço esquerdo"
    ],
    "pressao_arterial": "150/95 mmHg",
    "frequencia_cardiaca": "110 bpm",
    "historico_doencas": [
        "hipertensão"
    ],
    "medicamentos_em_uso": [
        "Losartana 50mg"
    ],
    "procedimentos_solicitados": [
        "ECG de urgência",
        "exames de enzimas cardíacas"
    ],
    "nivel_urgencia": "Alto"
}
